## Step 1: Data Audit

The objective of this notebook is only to understand the data. It will produce your first five outputs:
1. Dataset row counts
2. Column information
3. Missing-value summary
4. Basic data-quality information
5. Duplicate statistics

Do not normalize or modify the data yet.

### Cell 1 — Install/import

In [16]:
!git clone https://github.com/Ayushxsingh100/Amazon-ML-Challenge-2026.git

Cloning into 'Amazon-ML-Challenge-2026'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
^C


In [17]:
from pathlib import Path

PROJECT_DIR = Path("/content/Amazon-ML-Challenge-2026")

TRAIN_DIR = PROJECT_DIR / "data" / "train"
TEST_DIR = PROJECT_DIR / "data" / "test"

print("Project:", PROJECT_DIR)
print("Train:", TRAIN_DIR)
print("Test:", TEST_DIR)

Project: /content/Amazon-ML-Challenge-2026
Train: /content/Amazon-ML-Challenge-2026/data/train
Test: /content/Amazon-ML-Challenge-2026/data/test


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Cell 2 — Connect Google Drive

In [19]:
print("Project exists:", PROJECT_DIR.exists())
print("Train exists:", TRAIN_DIR.exists())
print("Test exists:", TEST_DIR.exists())

Project exists: True
Train exists: True
Test exists: True


### Cell 3 — Set your data directory

In [20]:
print("\n========== TRAIN FILES ==========")

for f in sorted(TRAIN_DIR.iterdir()):
    print(f.name)

print("\n========== TEST FILES ==========")

for f in sorted(TEST_DIR.iterdir()):
    print(f.name)


========== TRAIN FILES ==========
train_ground_truth.tsv
train_ground_truth.tsv.part_aa
train_ground_truth.tsv.part_ab
train_source1.tsv
train_source1.tsv.part_aa
train_source1.tsv.part_ab
train_source1.tsv.part_ac
train_source2.tsv
train_source2.tsv.part_aa
train_source2.tsv.part_ab
train_source2.tsv.part_ac
train_source2.tsv.part_ad
train_source2.tsv.part_ae
train_source2.tsv.part_af
train_source3.tsv
train_source3.tsv.part_aa
train_source3.tsv.part_ab
train_source3.tsv.part_ac
train_source3.tsv.part_ad
train_source3.tsv.part_ae
train_source3.tsv.part_af

========== TEST FILES ==========
test_source1.tsv
test_source1.tsv.part_aa
test_source1.tsv.part_ab
test_source2.tsv
test_source2.tsv.part_aa
test_source2.tsv.part_ab
test_source2.tsv.part_ac
test_source2.tsv.part_ad
test_source2.tsv.part_ae
test_source2.tsv.part_af
test_source3.tsv
test_source3.tsv.part_aa
test_source3.tsv.part_ab
test_source3.tsv.part_ac
test_source3.tsv.part_ad
test_source3.tsv.part_ae
test_source3.tsv.part_af


### Cell 4 — Automatically discover the chunks

In [21]:
from pathlib import Path

print("========== PROJECT STRUCTURE ==========")

for path in sorted(PROJECT_DIR.iterdir()):
    print(path.name)

print("\n========== TRAIN FILES ==========")

for path in sorted(TRAIN_DIR.iterdir()):
    print(path.name)

print("\n========== TEST FILES ==========")

for path in sorted(TEST_DIR.iterdir()):
    print(path.name)

========== PROJECT STRUCTURE ==========
.git
.gitignore
README.md
data
notebooks
reconstruct_data.sh
src

========== TRAIN FILES ==========
train_ground_truth.tsv
train_ground_truth.tsv.part_aa
train_ground_truth.tsv.part_ab
train_source1.tsv
train_source1.tsv.part_aa
train_source1.tsv.part_ab
train_source1.tsv.part_ac
train_source2.tsv
train_source2.tsv.part_aa
train_source2.tsv.part_ab
train_source2.tsv.part_ac
train_source2.tsv.part_ad
train_source2.tsv.part_ae
train_source2.tsv.part_af
train_source3.tsv
train_source3.tsv.part_aa
train_source3.tsv.part_ab
train_source3.tsv.part_ac
train_source3.tsv.part_ad
train_source3.tsv.part_ae
train_source3.tsv.part_af

========== TEST FILES ==========
test_source1.tsv
test_source1.tsv.part_aa
test_source1.tsv.part_ab
test_source2.tsv
test_source2.tsv.part_aa
test_source2.tsv.part_ab
test_source2.tsv.part_ac
test_source2.tsv.part_ad
test_source2.tsv.part_ae
test_source2.tsv.part_af
test_source3.tsv
test_source3.tsv.part_aa
test_source3.tsv.part

### Cell 5 — Display discovered parts

In [22]:
SCRIPT = PROJECT_DIR / "reconstruct_data.sh"

print(SCRIPT.read_text())

#!/usr/bin/env bash
# Amazon ML Challenge 2026 - Reconstruct Data TSV files from git split parts
set -e

echo "=== Reconstructing Amazon ML Challenge 2026 Data Files ==="

reconstruct_file() {
    local target="$1"
    local dir=$(dirname "$target")
    local base=$(basename "$target")
    
    if [ -f "$target" ]; then
        echo "[EXISTS] $target is already present."
    else
        echo "[MERGING] Combining parts for $target ..."
        cat "${target}.part_"* > "$target"
        echo "[DONE] Successfully created $target ($(du -h "$target" | cut -f1))"
    fi
}

# Train sources
reconstruct_file "data/train/train_source1.tsv"
reconstruct_file "data/train/train_source2.tsv"
reconstruct_file "data/train/train_source3.tsv"
reconstruct_file "data/train/train_ground_truth.tsv"

# Test sources
reconstruct_file "data/test/test_source1.tsv"
reconstruct_file "data/test/test_source2.tsv"
reconstruct_file "data/test/test_source3.tsv"

echo "=== All data files ready! ==="



### Cell 6 — Check chunk sizes

Perfect. The screenshots confirm the important thing we needed:

*   `part_aa` contains the header.
*   `part_ab`, `part_ac`, etc. contain only data rows.

Now continue from Cell 7.

### Cell 7 — Reconstruct the original TSV files

In [23]:
import os
import subprocess

# Move into the repository because reconstruct_data.sh
# uses relative paths such as data/train/...
os.chdir(PROJECT_DIR)
print("Current directory:")
print(Path.cwd())
print("\nRunning reconstruction script...\n")

result = subprocess.run(
    ["bash", "reconstruct_data.sh"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("\nReturn code:", result.returncode)

Current directory:
/content/Amazon-ML-Challenge-2026

Running reconstruction script...

=== Reconstructing Amazon ML Challenge 2026 Data Files ===
[EXISTS] data/train/train_source1.tsv is already present.
[EXISTS] data/train/train_source2.tsv is already present.
[EXISTS] data/train/train_source3.tsv is already present.
[EXISTS] data/train/train_ground_truth.tsv is already present.
[EXISTS] data/test/test_source1.tsv is already present.
[EXISTS] data/test/test_source2.tsv is already present.
[EXISTS] data/test/test_source3.tsv is already present.
=== All data files ready! ===


Return code: 0


### Cell 8 — Verify reconstructed files

In [24]:
print("========== TRAIN FILES ==========")
for f in sorted(TRAIN_DIR.iterdir()):
    print(f.name)
print("\n========== TEST FILES ==========")
for f in sorted(TEST_DIR.iterdir()):
    print(f.name)

========== TRAIN FILES ==========
train_ground_truth.tsv
train_ground_truth.tsv.part_aa
train_ground_truth.tsv.part_ab
train_source1.tsv
train_source1.tsv.part_aa
train_source1.tsv.part_ab
train_source1.tsv.part_ac
train_source2.tsv
train_source2.tsv.part_aa
train_source2.tsv.part_ab
train_source2.tsv.part_ac
train_source2.tsv.part_ad
train_source2.tsv.part_ae
train_source2.tsv.part_af
train_source3.tsv
train_source3.tsv.part_aa
train_source3.tsv.part_ab
train_source3.tsv.part_ac
train_source3.tsv.part_ad
train_source3.tsv.part_ae
train_source3.tsv.part_af

========== TEST FILES ==========
test_source1.tsv
test_source1.tsv.part_aa
test_source1.tsv.part_ab
test_source2.tsv
test_source2.tsv.part_aa
test_source2.tsv.part_ab
test_source2.tsv.part_ac
test_source2.tsv.part_ad
test_source2.tsv.part_ae
test_source2.tsv.part_af
test_source3.tsv
test_source3.tsv.part_aa
test_source3.tsv.part_ab
test_source3.tsv.part_ac
test_source3.tsv.part_ad
test_source3.tsv.part_ae
test_source3.tsv.part_af


### Cell 9 — Check reconstructed file sizes

In [25]:
all_reconstructed = [
    TRAIN_DIR / "train_source1.tsv",
    TRAIN_DIR / "train_source2.tsv",
    TRAIN_DIR / "train_source3.tsv",
    TRAIN_DIR / "train_ground_truth.tsv",
    TEST_DIR / "test_source1.tsv",
    TEST_DIR / "test_source2.tsv",
    TEST_DIR / "test_source3.tsv",
]
rows = []
for path in all_reconstructed:
    if path.exists():
        size_gb = path.stat().st_size / (1024 ** 3)
        rows.append({
            "file": path.name,
            "size_GB": round(size_gb, 3),
            "exists": True
        })
    else:
        rows.append({
            "file": path.name,
            "size_GB": None,
            "exists": False
        })
display(pd.DataFrame(rows))

,file,size_GB,exists
0,train_source1.tsv,0.196,True
1,train_source2.tsv,0.456,True
2,train_source3.tsv,0.469,True
3,train_ground_truth.tsv,0.118,True
4,test_source1.tsv,0.163,True
5,test_source2.tsv,0.474,True
6,test_source3.tsv,0.471,True


### Cell 10 — Verify the reconstructed headers

In [26]:
for path in all_reconstructed:
    if path.exists():
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            first_line = f.readline().strip()
        print("\n", path.name)
        print(first_line)


 train_source1.tsv
entity_id	business_name	business_address	country

 train_source2.tsv
entity_id	business_name	business_address	country

 train_source3.tsv
entity_id	business_name	business_address	country

 train_ground_truth.tsv
source1_entity_id	matched_entity_ids

 test_source1.tsv
entity_id	business_name	business_address	country

 test_source2.tsv
entity_id	business_name	business_address	country

 test_source3.tsv
entity_id	business_name	business_address	country


### Cell 11 — Load ONLY the headers

In [27]:
SOURCE_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]
GT_COLUMNS = [
    "source1_entity_id",
    "matched_entity_ids"
]
source_files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
}
gt_file = TRAIN_DIR / "train_ground_truth.tsv"
for name, path in source_files.items():
    df_sample = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=5
    )
    print("\n", name)
    print(df_sample.columns.tolist())
    display(df_sample)


 train_source1
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



 train_source2
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



 train_source3
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [28]:
gt_sample = pd.read_csv(
    gt_file,
    sep="\t",
    dtype=str,
    nrows=5
)
print("\ntrain_ground_truth")
print(gt_sample.columns.tolist())
display(gt_sample)


train_ground_truth
['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


### Cell 12 — Count rows efficiently

In [29]:
def count_data_rows(path):
    with open(path, "rb") as f:
        row_count = sum(1 for _ in f)
    # subtract header
    return row_count - 1
row_counts = []
for name, path in {
    **source_files,
    "train_ground_truth": gt_file
}.items():
    count = count_data_rows(path)
    row_counts.append({
        "dataset": name,
        "rows": count
    })
row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

,dataset,rows
0,train_source1,2206821
1,train_source2,5034616
2,train_source3,5285603
3,train_ground_truth,2206821


### Cell 13 — Column information

In [30]:
column_report = []
for name, path in source_files.items():
    sample = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=1000
    )
    for col in sample.columns:
        column_report.append({
            "dataset": name,
            "column": col,
            "dtype": str(sample[col].dtype),
            "sample_non_null": int(sample[col].notna().sum()),
            "sample_unique": int(sample[col].nunique())
        })
gt_sample = pd.read_csv(
    gt_file,
    sep="\t",
    dtype=str,
    nrows=1000
)
for col in gt_sample.columns:
    column_report.append({
        "dataset": "train_ground_truth",
        "column": col,
        "dtype": str(gt_sample[col].dtype),
        "sample_non_null": int(gt_sample[col].notna().sum()),
        "sample_unique": int(gt_sample[col].nunique())
    })
column_report_df = pd.DataFrame(column_report)
display(column_report_df)

,dataset,column,dtype,sample_non_null,sample_unique
0,train_source1,entity_id,object,1000,1000
1,train_source1,business_name,object,1000,1000
2,train_source1,business_address,object,1000,1000
3,train_source1,country,object,1000,2
4,train_source2,entity_id,object,1000,1000
5,train_source2,business_name,object,1000,1000
6,train_source2,business_address,object,965,965
7,train_source2,country,object,1000,2
8,train_source3,entity_id,object,1000,1000
9,train_source3,business_name,object,1000,1000


### Cell 14 — Missing-value audit

In [31]:
def missing_value_audit(
    path,
    columns,
    chunksize=100_000
):

    missing_counts = {col: 0 for col in columns}
    total_rows = 0
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=columns,
        chunksize=chunksize
    ):
        total_rows += len(chunk)
        for col in columns:
            missing_counts[col] += int(
                chunk[col].isna().sum()
            )
    rows = []
    for col in columns:
        count = missing_counts[col]
        rows.append({
            "column": col,
            "missing_count": count,
            "total_rows": total_rows,
            "missing_percentage": (
                count / total_rows * 100
            )
        })
    return pd.DataFrame(rows)

for name, path in source_files.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    display(
        missing_value_audit(
            path,
            SOURCE_COLUMNS
        )
    )


train_source1


,column,missing_count,total_rows,missing_percentage
0,entity_id,0,2206821,0.0
1,business_name,0,2206821,0.0
2,business_address,0,2206821,0.0
3,country,0,2206821,0.0



train_source2


,column,missing_count,total_rows,missing_percentage
0,entity_id,0,5034616,0.000000
1,business_name,2,5034616,0.000040
2,business_address,168967,5034616,3.356105
3,country,0,5034616,0.000000



train_source3


,column,missing_count,total_rows,missing_percentage
0,entity_id,0,5285603,0.000000
1,business_name,13,5285603,0.000246
2,business_address,175916,5285603,3.328211
3,country,0,5285603,0.000000


In [32]:
print("\n" + "=" * 70)
print("train_ground_truth")
print("=" * 70)
display(
    missing_value_audit(
        gt_file,
        GT_COLUMNS
    )
)


train_ground_truth


,column,missing_count,total_rows,missing_percentage
0,source1_entity_id,0,2206821,0.000000
1,matched_entity_ids,123247,2206821,5.584821


### Cell 15 — Blank-string audit

In [33]:
def blank_value_audit(
    path,
    columns,
    chunksize=100_000
):
    blank_counts = {col: 0 for col in columns}
    total_rows = 0
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=columns,
        keep_default_na=False,
        chunksize=chunksize
    ):
        total_rows += len(chunk)
        for col in columns:
            blank_counts[col] += int(
                chunk[col]
                .astype(str)
                .str.strip()
                .eq("")
                .sum()
            )
    rows = []
    for col in columns:
        count = blank_counts[col]
        rows.append({
            "column": col,
            "blank_count": count,
            "total_rows": total_rows,
            "blank_percentage": (
                count / total_rows * 100
            )
        })
    return pd.DataFrame(rows)

for name, path in source_files.items():
    print("\n", name)
    display(
        blank_value_audit(
            path,
            SOURCE_COLUMNS
        )
    )


 train_source1


,column,blank_count,total_rows,blank_percentage
0,entity_id,0,2206821,0.0
1,business_name,0,2206821,0.0
2,business_address,0,2206821,0.0
3,country,0,2206821,0.0



 train_source2


,column,blank_count,total_rows,blank_percentage
0,entity_id,0,5034616,0.000000
1,business_name,0,5034616,0.000000
2,business_address,168967,5034616,3.356105
3,country,0,5034616,0.000000



 train_source3


,column,blank_count,total_rows,blank_percentage
0,entity_id,0,5285603,0.000000
1,business_name,0,5285603,0.000000
2,business_address,175916,5285603,3.328211
3,country,0,5285603,0.000000


### Cell 16 — Duplicate entity ID analysis

In [34]:
def duplicate_id_audit(
    path,
    id_column="entity_id",
    chunksize=100_000
):
    seen = set()
    duplicate_ids = set()
    total_rows = 0
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[id_column],
        chunksize=chunksize
    ):
        for entity_id in chunk[id_column].dropna():
            total_rows += 1
            if entity_id in seen:
                duplicate_ids.add(entity_id)
            else:
                seen.add(entity_id)
    return {
        "total_rows": total_rows,
        "unique_ids": len(seen),
        "duplicate_ids": len(duplicate_ids),
        "duplicate_examples": list(duplicate_ids)[:10]
    }

duplicate_results = []
for name, path in source_files.items():
    result = duplicate_id_audit(path)
    duplicate_results.append({
        "dataset": name,
        "total_rows": result["total_rows"],
        "unique_ids": result["unique_ids"],
        "duplicate_ids": result["duplicate_ids"]
    })
duplicate_df = pd.DataFrame(duplicate_results)
display(duplicate_df)

,dataset,total_rows,unique_ids,duplicate_ids
0,train_source1,2206821,2206821,0
1,train_source2,5034616,5034616,0
2,train_source3,5285603,5285603,0


### Cell 17 — Validate source prefixes

In [35]:
def prefix_audit(
    path,
    expected_prefix,
    chunksize=100_000
):
    total = 0
    wrong = 0
    examples = []
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["entity_id"],
        chunksize=chunksize
    ):
        ids = chunk["entity_id"].dropna().astype(str)
        total += len(ids)
        bad = ~ids.str.startswith(expected_prefix)
        wrong += int(bad.sum())
        if bad.any() and len(examples) < 10:
            examples.extend(
                ids[bad].head(10 - len(examples)).tolist()
            )
    return {
        "total": total,
        "wrong_prefix": wrong,
        "examples": examples
    }

prefix_results = []
for name, path, prefix in [
    ("train_source1", source_files["train_source1"], "S1-"),
    ("train_source2", source_files["train_source2"], "S2-"),
    ("train_source3", source_files["train_source3"], "S3-"),
]:
    result = prefix_audit(
        path,
        prefix
    )
    prefix_results.append({
        "dataset": name,
        "expected_prefix": prefix,
        "total": result["total"],
        "wrong_prefix": result["wrong_prefix"],
        "examples": result["examples"]
    })
prefix_df = pd.DataFrame(prefix_results)
display(prefix_df)

,dataset,expected_prefix,total,wrong_prefix,examples
0,train_source1,S1-,2206821,0,[]
1,train_source2,S2-,5034616,0,[]
2,train_source3,S3-,5285603,0,[]


### Cell 18 — Country distribution

In [36]:
from collections import Counter
def country_distribution(
    path,
    chunksize=100_000
):
    counter = Counter()
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["country"],
        keep_default_na=False,
        chunksize=chunksize
    ):
        values = (
            chunk["country"]
            .astype(str)
            .str.strip()
        )
        counter.update(values)
    result = pd.DataFrame(
        counter.items(),
        columns=["country", "count"]
    )
    result["percentage"] = (
        result["count"]
        / result["count"].sum()
        * 100
    )
    return result.sort_values(
        "count",
        ascending=False
    )

for name, path in source_files.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    display(
        country_distribution(path).head(30)
    )


train_source1


,country,count,percentage
0,US,1323633,59.979174
1,India,883188,40.020826



train_source2


,country,count,percentage
1,US,3016817,59.921492
0,India,2017799,40.078508



train_source3


,country,count,percentage
0,US,3170056,59.975295
1,India,2115547,40.024705


### Cell 19 — Name/address length statistics

In [37]:
def string_length_statistics(
    path,
    column,
    chunksize=100_000
):
    lengths = []
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[column],
        keep_default_na=False,
        chunksize=chunksize
    ):
        values = (
            chunk[column]
            .astype(str)
        )
        lengths.extend(
            values.str.len().tolist()
        )
    lengths = np.array(lengths)
    return {
        "column": column,
        "rows": len(lengths),
        "min": int(lengths.min()),
        "mean": round(float(lengths.mean()), 2),
        "median": round(float(np.median(lengths)), 2),
        "p90": round(float(np.percentile(lengths, 90)), 2),
        "p95": round(float(np.percentile(lengths, 95)), 2),
        "p99": round(float(np.percentile(lengths, 99)), 2),
        "max": int(lengths.max())
    }

length_results = []
for name, path in source_files.items():
    for column in [
        "business_name",
        "business_address"
    ]:
        result = string_length_statistics(
            path,
            column
        )
        result["dataset"] = name
        length_results.append(result)
length_df = pd.DataFrame(length_results)
display(length_df)

,column,rows,min,mean,median,p90,p95,p99,max,dataset
0,business_name,2206821,3,24.03,24.0,34.0,37.0,42.0,105,train_source1
1,business_address,2206821,11,52.07,41.0,90.0,103.0,124.0,256,train_source1
2,business_name,5034616,2,25.10,25.0,37.0,40.0,48.0,104,train_source2
3,business_address,5034616,0,46.23,37.0,83.0,96.0,118.0,249,train_source2
4,business_name,5285603,2,25.20,25.0,37.0,42.0,50.0,123,train_source3
5,business_address,5285603,0,46.71,42.0,77.0,91.0,115.0,240,train_source3


### Cell 20 — Save all Step-1 outputs

In [38]:
AUDIT_DIR = PROJECT_DIR / "outputs" / "person1_step1"
AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
print("Output directory:")
print(AUDIT_DIR)

row_counts_df.to_csv(
    AUDIT_DIR / "dataset_row_counts.tsv",
    sep="\t",
    index=False
)
column_report_df.to_csv(
    AUDIT_DIR / "column_information.tsv",
    sep="\t",
    index=False
)
duplicate_df.to_csv(
    AUDIT_DIR / "duplicate_id_summary.tsv",
    sep="\t",
    index=False
)
prefix_df.to_csv(
    AUDIT_DIR / "id_prefix_summary.tsv",
    sep="\t",
    index=False
)
length_df.to_csv(
    AUDIT_DIR / "string_length_statistics.tsv",
    sep="\t",
    index=False
)
for name, path in source_files.items():
    country_df = country_distribution(path)
    country_df.to_csv(
        AUDIT_DIR / f"{name}_country_distribution.tsv",
        sep="\t",
        index=False
    )
print("Step-1 reports saved.")

Output directory:
/content/Amazon-ML-Challenge-2026/outputs/person1_step1
Step-1 reports saved.


In [39]:
def inspect_chunk_headers(directory, prefix):
    parts = sorted(directory.glob(prefix + ".tsv.part_*"))

    print(f"\n{'='*80}")
    print(prefix)
    print(f"{'='*80}")

    for part in parts:
        with open(part, "r", encoding="utf-8", errors="replace") as f:
            first_line = f.readline().strip()

        print(f"\n{part.name}")
        print(first_line[:300])


for prefix in [
    "train_source1",
    "train_source2",
    "train_source3",
    "train_ground_truth"
]:
    inspect_chunk_headers(TRAIN_DIR, prefix)


train_source1

train_source1.tsv.part_aa
entity_id	business_name	business_address	country

train_source1.tsv.part_ab
gar, Vijayawada, Krishna, Andhra Pradesh	India

train_source1.tsv.part_ac
nkcity Impex Private Limited	Flat No.11 Dwarka Apts 467/C Shivaji Nagar, Pune, Maharashtra	India

train_source2

train_source2.tsv.part_aa
entity_id	business_name	business_address	country

train_source2.tsv.part_ab
170 HIGLHAND AVENUE, HAMBURG, NY	US

train_source2.tsv.part_ac
9984094	Durham Womens Health Integrated Physicians Corp	3960 RIVER STONE ROAD, DURHAM, NC	US

train_source2.tsv.part_ad
-446836062	New Constructions  Hotel Private Limited	KHASRA NO 270, VILL, RASULPUR, MOJA BHARAMPUR, ETMADPUR, Uttar Pradesh	India

train_source2.tsv.part_ae
LE, KY	US

train_source2.tsv.part_af
मॉडर्न डेवलपर्स	Maharashtra, PUNE, KUMAR KSHITIJ FLAT B A-304SAHAKAR NAGAR -2	India

train_source3

train_source3.tsv.part_aa
entity_id	business_name	business_address	country

train_source3.tsv.part_ab
Fund Holdings	6

### STEP 1A — Run Cell 17: ID prefix validation

In [40]:
def prefix_audit(
    path,
    expected_prefix,
    chunksize=100_000
):
    total = 0
    wrong = 0
    examples = []
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["entity_id"],
        chunksize=chunksize
    ):
        ids = chunk["entity_id"].dropna().astype(str)
        total += len(ids)
        bad = ~ids.str.startswith(expected_prefix)
        wrong += int(bad.sum())
        if bad.any() and len(examples) < 10:
            examples.extend(
                ids[bad].head(10 - len(examples)).tolist()
            )
    return {
        "total": total,
        "wrong_prefix": wrong,
        "examples": examples
    }
prefix_results = []
for name, path, prefix in [
    ("train_source1", source_files["train_source1"], "S1-"),
    ("train_source2", source_files["train_source2"], "S2-"),
    ("train_source3", source_files["train_source3"], "S3-"),
]:
    result = prefix_audit(path, prefix)
    prefix_results.append({
        "dataset": name,
        "expected_prefix": prefix,
        "total": result["total"],
        "wrong_prefix": result["wrong_prefix"],
        "examples": result["examples"]
    })
prefix_df = pd.DataFrame(prefix_results)
display(prefix_df)

,dataset,expected_prefix,total,wrong_prefix,examples
0,train_source1,S1-,2206821,0,[]
1,train_source2,S2-,5034616,0,[]
2,train_source3,S3-,5285603,0,[]


We want:

`wrong_prefix = 0`

for all three.

### STEP 1B — Also check TEST IDs

In [41]:
test_source_files = {
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}
test_prefix_results = []
for name, path, prefix in [
    ("test_source1", test_source_files["test_source1"], "S1-"),
    ("test_source2", test_source_files["test_source2"], "S2-"),
    ("test_source3", test_source_files["test_source3"], "S3-"),
]:
    result = prefix_audit(path, prefix)
    test_prefix_results.append({
        "dataset": name,
        "expected_prefix": prefix,
        "total": result["total"],
        "wrong_prefix": result["wrong_prefix"],
        "examples": result["examples"]
    })
test_prefix_df = pd.DataFrame(test_prefix_results)
display(test_prefix_df)

,dataset,expected_prefix,total,wrong_prefix,examples
0,test_source1,S1-,1732544,0,[]
1,test_source2,S2-,4887273,0,[]
2,test_source3,S3-,5082316,0,[]


### STEP 1C — Count the TEST datasets

In [42]:
test_row_counts = []
for name, path in test_source_files.items():
    count = count_data_rows(path)
    test_row_counts.append({
        "dataset": name,
        "rows": count
    })
test_row_counts_df = pd.DataFrame(test_row_counts)
display(test_row_counts_df)

,dataset,rows
0,test_source1,1732544
1,test_source2,4887273
2,test_source3,5082316


This gives us:

`test_source1` → ?
`test_source2` → ?
`test_source3` → ?

These numbers are extremely important for understanding the eventual candidate-generation problem.

### STEP 1D — Run the string-length analysis

In [43]:
import numpy as np
def string_length_statistics(
    path,
    column,
    chunksize=100_000
):
    lengths = []
    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[column],
        keep_default_na=False,
        chunksize=chunksize
    ):
        values = chunk[column].astype(str)
        lengths.extend(values.str.len().tolist())
    lengths = np.array(lengths)
    return {
        "column": column,
        "rows": len(lengths),
        "min": int(lengths.min()),
        "mean": round(float(lengths.mean()), 2),
        "median": round(float(np.median(lengths)), 2),
        "p90": round(float(np.percentile(lengths, 90)), 2),
        "p95": round(float(np.percentile(lengths, 95)), 2),
        "p99": round(float(np.percentile(lengths, 99)), 2),
        "max": int(lengths.max())
    }
length_results = []
for name, path in source_files.items():
    for column in [
        "business_name",
        "business_address"
    ]:
        result = string_length_statistics(
            path,
            column
        )
        result["dataset"] = name
        length_results.append(result)
length_df = pd.DataFrame(length_results)
display(length_df)

,column,rows,min,mean,median,p90,p95,p99,max,dataset
0,business_name,2206821,3,24.03,24.0,34.0,37.0,42.0,105,train_source1
1,business_address,2206821,11,52.07,41.0,90.0,103.0,124.0,256,train_source1
2,business_name,5034616,2,25.10,25.0,37.0,40.0,48.0,104,train_source2
3,business_address,5034616,0,46.23,37.0,83.0,96.0,118.0,249,train_source2
4,business_name,5285603,2,25.20,25.0,37.0,42.0,50.0,123,train_source3
5,business_address,5285603,0,46.71,42.0,77.0,91.0,115.0,240,train_source3


This helps us decide later whether character n-grams, token matching, edit distance, etc. make sense.

### STEP 1E — Save the missing-value results properly

In [44]:
missing_reports = {}
for name, path in source_files.items():
    report = missing_value_audit(
        path,
        SOURCE_COLUMNS
    )
    missing_reports[name] = report
    report.to_csv(
        AUDIT_DIR / f"{name}_missing_values.tsv",
        sep="\t",
        index=False
    )
gt_missing_report = missing_value_audit(
    gt_file,
    GT_COLUMNS
)
gt_missing_report.to_csv(
    AUDIT_DIR / "train_ground_truth_missing_values.tsv",
    sep="\t",
    index=False
)
print("Missing-value reports saved.")

Missing-value reports saved.


### STEP 1F — Save the blank-value results

In [45]:
blank_reports = {}
for name, path in source_files.items():
    report = blank_value_audit(
        path,
        SOURCE_COLUMNS
    )
    blank_reports[name] = report
    report.to_csv(
        AUDIT_DIR / f"{name}_blank_values.tsv",
        sep="\t",
        index=False
    )
gt_blank_report = blank_value_audit(
    gt_file,
    GT_COLUMNS
)
gt_blank_report.to_csv(
    AUDIT_DIR / "train_ground_truth_blank_values.tsv",
    sep="\t",
    index=False
)
print("Blank-value reports saved.")

Blank-value reports saved.


### STEP 1G — Save everything

In [46]:
# Re-define and ensure AUDIT_DIR exists, as this is a consolidated save cell
AUDIT_DIR = PROJECT_DIR / "outputs" / "person1_step1"
AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save row counts
row_counts_df.to_csv(
    AUDIT_DIR / "train_dataset_row_counts.tsv",
    sep="\t",
    index=False
)
test_row_counts_df.to_csv(
    AUDIT_DIR / "test_dataset_row_counts.tsv",
    sep="\t",
    index=False
)
# Save schema
column_report_df.to_csv(
    AUDIT_DIR / "column_information.tsv",
    sep="\t",
    index=False
)
# Save duplicate information
duplicate_df.to_csv(
    AUDIT_DIR / "duplicate_id_summary.tsv",
    sep="\t",
    index=False
)
# Save prefix checks
prefix_df.to_csv(
    AUDIT_DIR / "train_id_prefix_summary.tsv",
    sep="\t",
    index=False
)
test_prefix_df.to_csv(
    AUDIT_DIR / "test_id_prefix_summary.tsv",
    sep="\t",
    index=False
)
# Save string statistics
length_df.to_csv(
    AUDIT_DIR / "string_length_statistics.tsv",
    sep="\t",
    index=False
)
# Save country distributions
for name, path in source_files.items():
    country_df = country_distribution(path)
    country_df.to_csv(
        AUDIT_DIR / f"{name}_country_distribution.tsv",
        sep="\t",
        index=False
    )
print("========================================")
print("STEP 1 AUDIT COMPLETE")
print("========================================")
print("Output directory:")
print(AUDIT_DIR)

STEP 1 AUDIT COMPLETE
Output directory:
/content/Amazon-ML-Challenge-2026/outputs/person1_step1


### STEP 1H — Check the output folder

In [47]:
print("Files generated:\n")
for f in sorted(AUDIT_DIR.iterdir()):
    print(f.name)

Files generated:

column_information.tsv
dataset_row_counts.tsv
duplicate_id_summary.tsv
id_prefix_summary.tsv
string_length_statistics.tsv
test_dataset_row_counts.tsv
test_id_prefix_summary.tsv
train_dataset_row_counts.tsv
train_ground_truth_blank_values.tsv
train_ground_truth_missing_values.tsv
train_id_prefix_summary.tsv
train_source1_blank_values.tsv
train_source1_country_distribution.tsv
train_source1_missing_values.tsv
train_source2_blank_values.tsv
train_source2_country_distribution.tsv
train_source2_missing_values.tsv
train_source3_blank_values.tsv
train_source3_country_distribution.tsv
train_source3_missing_values.tsv


You should now have roughly:

```
person1_step1/
│
├── train_dataset_row_counts.tsv
├── test_dataset_row_counts.tsv
├── column_information.tsv
│
├── train_source1_missing_values.tsv
├── train_source2_missing_values.tsv
├── train_source3_missing_values.tsv
├── train_ground_truth_missing_values.tsv
│
├── train_source1_blank_values.tsv
├── train_source2_blank_values.tsv
├── train_source3_blank_values.tsv
├── train_ground_truth_blank_values.tsv
│
├── duplicate_id_summary.tsv
├── train_id_prefix_summary.tsv
├── test_id_prefix_summary.tsv
├── string_length_statistics.tsv
│
├── train_source1_country_distribution.tsv
├── train_source2_country_distribution.tsv
└── train_source3_country_distribution.tsv
```